In [1]:
# Stage 01: Data Inspection

from pathlib import Path
from zipfile import ZipFile, is_zipfile

# Automatically detect project directory
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

ZIP_PATH = PROJECT_DIR / "data" / "raw" / "FLAT_RCL_POST_2010.zip"

print("Project:", PROJECT_DIR)
print("Dataset exists:", ZIP_PATH.exists())

if ZIP_PATH.exists():
    print("Size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))
    print("Valid ZIP:", is_zipfile(ZIP_PATH))

    if is_zipfile(ZIP_PATH):
        with ZipFile(ZIP_PATH) as archive:
            print("\nFiles inside ZIP:")
            for file in archive.infolist():
                print(file.filename, "|", file.file_size, "bytes")


Project: C:\Users\faisa\Desktop\NLP Project
Dataset exists: True
Size (MB): 14.26
Valid ZIP: True

Files inside ZIP:
FLAT_RCL_POST_2010.txt | 313864373 bytes


In [2]:

# ==================================================
# Inspect Dataset Structure
# ==================================================

import csv
import io
from zipfile import ZipFile

# Read first record without extracting the ZIP
with ZipFile(ZIP_PATH, "r") as archive:

    txt_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".txt")
    )

    with archive.open(txt_name) as raw_file:
        with io.TextIOWrapper(
            raw_file,
            encoding="latin-1",
            newline=""
        ) as text_file:

            reader = csv.reader(
                text_file,
                delimiter="\t",
                quotechar='"'
            )

            first_row = next(reader)

print("Number of columns:", len(first_row))

print("\nFirst record:")
for index, value in enumerate(first_row, start=1):
    print(f"Column {index}: {value[:100]}")


# Inspect important fields in data dictionary
dictionary_path = (
    PROJECT_DIR / "data" / "raw" / "RCL_dictionary.txt"
)

print("\n--- Required Column Definitions ---")

if dictionary_path.exists():
    content = dictionary_path.read_text(
        encoding="latin-1",
        errors="replace"
    )

    required_fields = [
        "CAMPNO",
        "COMPNAME",
        "DESC_DEFECT",
        "RCLTYPECD"
    ]

    for line in content.splitlines():
        if any(field in line.upper() for field in required_fields):
            print(line)
else:
    print("Data dictionary not found.")


Number of columns: 29

First record:
Column 1: 81716
Column 2: 10V080000
Column 3: GEM
Column 4: e6
Column 5: 2009
Column 6: 
Column 7: PARKING BRAKE
Column 8: CHRYSLER GROUP GLOBAL ELECTRIC MOTORCARS
Column 9: 20091208
Column 10: 20091215
Column 11: V
Column 12: 142
Column 13: 20100310
Column 14: MFR
Column 15: GEM WAEV LLC
Column 16: 20100304
Column 17: 20100304
Column 18: 
Column 19: 
Column 20: GEM IS RECALLING CERTAIN MODEL YEAR 2009 NEIGHBORHOOD ELECTRIC VEHICLES.  THE PARKING BRAKE CABLE MA
Column 21: THE VEHICLE COULD ROLL AWAY ON A HILL CREATING A RISK OF CRASH OR INJURY.
Column 22: DEALERS WILL REPLACE THE PARKING BRAKE LEVER ASSEMBLY FREE OF CHARGE.  THE SAFETY RECALL BEGAN ON MA
Column 23: OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY TRAFFIC SAFETY ADMINISTRATION'S VEHICLE SAFETY HOTLINE 
Column 24: 000035299000847926000000115
Column 25: 
Column 26: 
Column 27: 
Column 28: No
Column 29: No

--- Required Column Definitions ---
2        CAMPNO              CHAR(12)    NHTSA R

In [3]:

# ==================================================
# Load Dataset Sample Using Pandas
# ==================================================

import pandas as pd
from zipfile import ZipFile

# Read the first 1000 records
with ZipFile(ZIP_PATH, "r") as archive:

    txt_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".txt")
    )

    with archive.open(txt_name) as file:

        df = pd.read_csv(
            file,
            sep="\t",
            header=None,
            encoding="latin-1",
            dtype=str,
            keep_default_na=False,
            nrows=1000
        )

# Confirm dataset structure
print("Dataset Shape:", df.shape)
print("Number of Rows:", len(df))
print("Number of Columns:", len(df.columns))

assert df.shape[1] == 29, "Unexpected number of columns!"

# Select the required columns
df_sample = df[[1, 6, 10, 19]].copy()

df_sample.columns = [
    "CAMPNO",
    "COMPNAME",
    "RCLTYPECD",
    "DESC_DEFECT"
]

# Display first 5 records
print("\nFirst 5 Records:")
display(df_sample.head())

# Check record types
print("\nRecall Type Distribution:")
print(df_sample["RCLTYPECD"].value_counts())

# Check common component labels
print("\nTop 10 Components:")
print(df_sample["COMPNAME"].value_counts().head(10))


Dataset Shape: (1000, 29)
Number of Rows: 1000
Number of Columns: 29

First 5 Records:


,CAMPNO,COMPNAME,RCLTYPECD,DESC_DEFECT
0,10V080000,PARKING BRAKE,V,GEM IS RECALLING CERTAIN MODEL YEAR 2009 NEIGH...
1,10V080000,PARKING BRAKE,V,GEM IS RECALLING CERTAIN MODEL YEAR 2009 NEIGH...
2,10V484000,EQUIPMENT,V,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...
3,10V484000,EQUIPMENT,V,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...
4,10V589000,ELECTRICAL SYSTEM:WIRING:FUSES AND CIRCUIT BRE...,V,DUTCHMEN IS RECALLING CERTAIN MODEL YEAR 2011 ...



Recall Type Distribution:
RCLTYPECD
V    918
E     64
C     11
T      7
Name: count, dtype: int64

Top 10 Components:
COMPNAME
EQUIPMENT                                        227
EQUIPMENT ADAPTIVE/MOBILITY                      115
ENGINE AND ENGINE COOLING                         87
STEERING                                          72
VEHICLE SPEED CONTROL                             37
SERVICE BRAKES, HYDRAULIC                         37
SEATS                                             29
VISIBILITY:POWER WINDOW DEVICES AND CONTROLS      26
POWER TRAIN:AXLE ASSEMBLY                         24
SERVICE BRAKES, HYDRAULIC:PEDALS AND LINKAGES     22
Name: count, dtype: int64


In [4]:

# ==================================================
# Full Dataset Analysis
# ==================================================

import pandas as pd
from zipfile import ZipFile
from collections import Counter

total_rows = 0
vehicle_rows = 0

type_counts = Counter()
component_counts = Counter()

unique_campaigns = set()

missing_description = 0
missing_component = 0
missing_campaign = 0

columns = [
    "CAMPNO",
    "COMPNAME",
    "RCLTYPECD",
    "DESC_DEFECT"
]

with ZipFile(ZIP_PATH, "r") as archive:

    txt_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".txt")
    )

    with archive.open(txt_name) as file:

        reader = pd.read_csv(
            file,
            sep="\t",
            header=None,
            usecols=[1, 6, 10, 19],
            encoding="latin-1",
            dtype=str,
            keep_default_na=False,
            chunksize=25000
        )

        for chunk in reader:

            chunk.columns = columns

            for col in columns:
                chunk[col] = chunk[col].str.strip()

            total_rows += len(chunk)

            type_counts.update(
                chunk["RCLTYPECD"].value_counts().to_dict()
            )

            # Vehicle recalls only
            vehicles = chunk[
                chunk["RCLTYPECD"].str.upper() == "V"
            ]

            vehicle_rows += len(vehicles)

            missing_description += (
                vehicles["DESC_DEFECT"] == ""
            ).sum()

            missing_component += (
                vehicles["COMPNAME"] == ""
            ).sum()

            missing_campaign += (
                vehicles["CAMPNO"] == ""
            ).sum()

            unique_campaigns.update(
                vehicles.loc[
                    vehicles["CAMPNO"] != "", "CAMPNO"
                ].unique()
            )

            component_counts.update(
                vehicles["COMPNAME"]
                .str.upper()
                .value_counts()
                .to_dict()
            )

# ==================================================
# Results
# ==================================================

print("=" * 55)
print("FULL DATASET ANALYSIS")
print("=" * 55)

print("Total records:", total_rows)
print("Vehicle records:", vehicle_rows)
print("Unique vehicle campaigns:", len(unique_campaigns))

print("\nRecall Types:")
for name, count in type_counts.most_common():
    print(name, ":", count)

print("\nMissing Values (Vehicle Records):")
print("Missing descriptions:", missing_description)
print("Missing components:", missing_component)
print("Missing campaign numbers:", missing_campaign)

print("\nTop 20 Vehicle Components:")
for component, count in component_counts.most_common(20):
    print(f"{component}: {count}")

# ==================================================
# Inspect our five target component groups
# ==================================================

targets = [
    "ELECTRICAL SYSTEM",
    "AIR BAGS",
    "SERVICE BRAKES",
    "STEERING",
    "FUEL SYSTEM"
]

print("\n" + "=" * 55)
print("FIVE TARGET COMPONENT GROUPS")
print("=" * 55)

for target in targets:

    matches = [
        (name, count)
        for name, count in component_counts.items()
        if name.startswith(target)
    ]

    matches.sort(key=lambda x: x[1], reverse=True)

    print(f"\n{target}")
    print("Matching records:", sum(n for _, n in matches))
    print("Distinct label names:", len(matches))

    for name, count in matches[:5]:
        print(f"  {name}: {count}")


FULL DATASET ANALYSIS
Total records: 248239
Vehicle records: 221485
Unique vehicle campaigns: 13764

Recall Types:
V : 221485
T : 20288
E : 6006
C : 460

Missing Values (Vehicle Records):
Missing descriptions: 0
Missing components: 0
Missing campaign numbers: 0

Top 20 Vehicle Components:
ELECTRICAL SYSTEM:SOFTWARE: 20500
COMMUNICATION: AUTO CRASH NOTIFICATION: 16688
EQUIPMENT: 9255
FUEL SYSTEM, GASOLINE:DELIVERY:FUEL PUMP: 8324
BACK OVER PREVENTION:DISPLAY FUNCTION: 7897
ELECTRICAL SYSTEM: INSTRUMENT CLUSTER/PANEL: 5267
ELECTRICAL SYSTEM: 4341
AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE: 4032
EQUIPMENT:OTHER:LABELS: 3877
BACK OVER PREVENTION: SENSING SYSTEM: CAMERA: 3768
EQUIPMENT ADAPTIVE/MOBILITY: 3608
ELECTRICAL SYSTEM:WIRING: 3094
AIR BAGS:FRONTAL:DRIVER SIDE:INFLATOR MODULE: 3053
EQUIPMENT:RECREATIONAL VEHICLE/TRAILER: 3046
SEATS: 2279
EXTERIOR LIGHTING:BRAKE LIGHTS: 2004
EXTERIOR LIGHTING: 1913
SEAT BELTS: 1794
AIR BAGS:FRONTAL: 1771
SEAT BELTS:REAR/OTHER:RETRACTOR: 1743

FI

In [5]:

# ==================================================
# Campaign-Level Analysis
# ==================================================

from collections import defaultdict, Counter
from zipfile import ZipFile
import pandas as pd

TARGET_CLASSES = {
    "ELECTRICAL SYSTEM": "Electrical System",
    "AIR BAGS": "Air Bags",
    "SERVICE BRAKES": "Service Brakes",
    "STEERING": "Steering",
    "FUEL SYSTEM": "Fuel System"
}

campaign_classes = defaultdict(set)
target_record_counts = Counter()

with ZipFile(ZIP_PATH, "r") as archive:

    txt_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".txt")
    )

    with archive.open(txt_name) as file:

        reader = pd.read_csv(
            file,
            sep="\t",
            header=None,
            usecols=[1, 6, 10, 19],
            encoding="latin-1",
            dtype=str,
            keep_default_na=False,
            chunksize=25000
        )

        for chunk in reader:

            chunk.columns = [
                "CAMPNO",
                "COMPNAME",
                "RCLTYPECD",
                "DESC_DEFECT"
            ]

            # Keep vehicle recalls only
            vehicles = chunk[
                chunk["RCLTYPECD"].str.strip().str.upper() == "V"
            ].copy()

            vehicles["COMPNAME"] = (
                vehicles["COMPNAME"].str.strip().str.upper()
            )

            vehicles["CAMPNO"] = (
                vehicles["CAMPNO"].str.strip()
            )

            # Match component groups
            for prefix, target in TARGET_CLASSES.items():

                matched = vehicles[
                    vehicles["COMPNAME"].str.startswith(prefix)
                ]

                target_record_counts[target] += len(matched)

                for campaign in matched["CAMPNO"]:
                    if campaign:
                        campaign_classes[campaign].add(target)

# Campaigns with exactly one target class
valid_campaigns = {
    campaign: next(iter(labels))
    for campaign, labels in campaign_classes.items()
    if len(labels) == 1
}

# Campaigns with multiple target classes
conflicting_campaigns = {
    campaign: labels
    for campaign, labels in campaign_classes.items()
    if len(labels) > 1
}

valid_counts = Counter(valid_campaigns.values())

print("=" * 55)
print("CAMPAIGN-LEVEL ANALYSIS")
print("=" * 55)

print("Matched target records:", sum(target_record_counts.values()))
print("Campaigns with target classes:", len(campaign_classes))
print("Campaigns with one target class:", len(valid_campaigns))
print("Conflicting campaigns:", len(conflicting_campaigns))

print("\nValid Campaigns by Class:")

for label in TARGET_CLASSES.values():
    print(f"{label}: {valid_counts[label]}")

print("\nExamples of Conflicting Campaigns:")

for campaign, labels in list(conflicting_campaigns.items())[:5]:
    print(campaign, "=>", sorted(labels))


CAMPAIGN-LEVEL ANALYSIS
Matched target records: 89857
Campaigns with target classes: 5261
Campaigns with one target class: 5096
Conflicting campaigns: 165

Valid Campaigns by Class:
Electrical System: 1896
Air Bags: 789
Service Brakes: 982
Steering: 691
Fuel System: 738

Examples of Conflicting Campaigns:
12V448000 => ['Electrical System', 'Service Brakes']
12V197000 => ['Electrical System', 'Service Brakes']
12V042000 => ['Electrical System', 'Service Brakes']
12V076000 => ['Electrical System', 'Fuel System']
12V349000 => ['Electrical System', 'Service Brakes']
